# Colabで検証済みの高速版：固定ソース・再開・品質評価
全件解析が実行中ならF4は実行しないでください。既存結果から品質評価する場合はF1→F2→F5→F6。新たな抽出は任意です。

## F1 固定ソース取得・ハッシュ照合・合成テスト
署名対象のソースを変更せず取得します。実牌譜やモデルは実行しません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,json,subprocess,urllib.request,hashlib
COMMIT='860605cf0f35f376be9dbd79763775c4300f11b4'
BASE=f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{COMMIT}/research_v2/fast_release_20261126'
HOME=Path('/tmp')/('received_fast_'+COMMIT[:12])
SOURCE=HOME/'snapshot/research_v2'
manifest=json.loads(urllib.request.urlopen(BASE+'/source_manifest.json').read())
for relative,expected in manifest.items():
    p=HOME/'snapshot'/relative;p.parent.mkdir(parents=True,exist_ok=True)
    content=urllib.request.urlopen(BASE+'/snapshot/'+relative).read()
    if hashlib.sha256(content).hexdigest()!=expected: raise ValueError('固定ソース不一致: '+relative)
    if p.exists() and p.read_bytes()!=content: raise ValueError('ローカル固定版が変更されています: '+relative)
    if not p.exists():p.write_bytes(content)
for name in ['quality_supplement.py','test_supplement.py']:
    (HOME/name).write_bytes(urllib.request.urlopen(BASE+'/'+name).read())
DEPS=HOME/'deps';stamp=DEPS/'installed'
if not stamp.exists():
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(DEPS),'-r',str(SOURCE/'features_100/requirements.txt')])
    stamp.write_text(COMMIT)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[DEPS,HOME,SOURCE,SOURCE/'choices',SOURCE/'smoothing'])),PYTHONIOENCODING='utf-8',MPLBACKEND='Agg')
subprocess.check_call([sys.executable,'-m','unittest','discover','-s',str(SOURCE/'features_100/tests'),'-t',str(SOURCE),'-q'],env=ENV)
subprocess.check_call([sys.executable,'-m','unittest','test_supplement','-q'],env=ENV,cwd=HOME)
subprocess.run(['apt-get','install','-y','-qq','fonts-noto-cjk'],check=False,stdout=subprocess.DEVNULL)
print('固定ソース照合完了。実牌譜の検証完了ではありません。')


## F2 入力・出力・既存全件結果の指定
OUTPUT_ROOTとcache_shardsは受領設定を維持。探索末とその根拠は自分で設定し、F5で両群の窓数を確認してください。過去に閲覧した期間は独立評価には戻せません。

In [ ]:
ROOT=Path('/content/drive/MyDrive/卒論')
CONFIG=dict(run_dir=str(ROOT/'research_v2_runs/20261105T071722Z_5da6c9ad'),
    output_root=str(ROOT/'research_v2_features_100_fast'),player='しょぼんｎ',key='shobon',
    zip_override=None,pilot_per_group=3,pilot_ids=None,workers=2,cache_shards=True,
    exploration_end_hour='',exploration_boundary_reason='')
EXTRACTION_DIR='' # 全件解析完了フォルダ（complete.jsonがある日時フォルダ）を設定
MODE='B' # 追加少数検証はA。実行中のBとは同時実行しない。
CFG=HOME/'config.json';RECEIPT=HOME/('preview_'+MODE+'.json')
def save_config():CFG.write_text(json.dumps(CONFIG,ensure_ascii=False),encoding='utf8')
def step(action,extra=()):
    save_config()
    subprocess.check_call([sys.executable,'-m','features_100.cli',action,str(CFG),'--mode',MODE,'--receipt',str(RECEIPT),*extra],env=ENV)
save_config()


## F3 任意：再開前の署名・件数確認
F4で再開する場合だけ実行。既存キャッシュとソースの整合性は各キャッシュ読込時にも検証します。ロック・一時ファイルを自動削除しません。

In [ ]:
step('preview')


## F4 任意：中断後の再開／少数追加抽出
実行中の全件解析がある場合は実行しない。成功キャッシュを再利用し、失敗を除外して窓を詰めません。残存ロックや署名不一致があればREADMEの手順に従い原因を確認してください。

In [ ]:
RUN_EXTRACTION=False
if RUN_EXTRACTION:
    step('extract')
    EXTRACTION_DIR=json.loads(RECEIPT.with_suffix('.result.json').read_text())['output']
else:print('抽出をスキップ。保存済み全件結果からF5〜F6を実行できます。')


## F5 探索境界の確認（ZIP再解析なし）
探索期間末をF2に設定してから実行。両群の窓数があることは必要条件であり、十分な行動機会を保証しません。境界を性能が良くなるように選ばないでください。

In [ ]:
save_config()
if not CONFIG['exploration_end_hour']:raise ValueError('F2の探索期間末と根拠を設定してください')
subprocess.check_call([sys.executable,str(HOME/'quality_supplement.py'),'boundary',str(CFG)],env=ENV)


## F6 全件抽出の品質評価＋群別相関
既存署名・CSVハッシュ・C5のID/段位/座席/XML対応を検証後、新しい日時フォルダに出力。分母0・unknownは補完しません。モデル学習なし。

In [ ]:
if not EXTRACTION_DIR:raise ValueError('F2で全件抽出完了フォルダを指定してください')
save_config()
subprocess.check_call([sys.executable,str(HOME/'quality_supplement.py'),'quality',str(CFG),'--extraction-dir',str(EXTRACTION_DIR)],env=ENV)
print('出力:',Path(CONFIG['output_root'])/'quality','の新しい日時フォルダ')


## 確認する成果物
group_statistics.csv、feature_comparison.csv、period_and_exposure_statistics.csv、sampling_noise_reference.csv、feature_correlations.csv、partition_counts.csv、boundary_rationale.json、report_ja.md、figures/。
小分母・局内依存・時間と段位の交絡に注意。高相関や低分散だけで採否を決めません。副露直前情報・Z独立検算・未観測候補の監査は別途残っています。